# Glass crack models: Colab session

Run cells 1-4 at the start of **every** session, then **one** training cell. Everything heavy lives in repo scripts; this notebook only sets up paths and calls them.

- Data: `bundle_colab.zip` on Drive is copied to `/content` and unzipped (fast local disk).
- Outputs (checkpoints, evals, `results/results.csv`) go to Drive, so a disconnect loses at most one epoch. Re-running the same training cell resumes.

In [ ]:
# 1. GPU and preinstalled torch (note the GPU type and torch version for the run log)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print('torch', torch.__version__, 'torchvision', torchvision.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive and paths
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/glass_crack'          # holds bundle_colab.zip; outputs/ is created here
REPO_URL = 'https://github.com/<you>/cracked.git'      # set once the GitHub repo exists
os.environ['DATA_ROOT'] = '/content/bundle'
os.environ['OUT_ROOT'] = f'{DRIVE}/outputs'
os.makedirs(os.environ['OUT_ROOT'], exist_ok=True)

In [ ]:
# 3. Code and packages (Colab's own torch is kept; pins from requirements.md)
%cd /content
!test -d cracked && git -C cracked pull --ff-only || git clone {REPO_URL} cracked
%cd /content/cracked
!pip install -q ultralytics==8.4.177 segmentation-models-pytorch==0.5.0 timm==1.0.30 transformers==5.19.0 albumentations==2.0.8 pycocotools==2.0.11 scikit-image==0.26.0 shapely==2.2.0
!git log -1 --oneline

In [ ]:
# 4. Data: copy the archive to local disk, unzip, build the linear_defect set and weighting lists
import os
if not os.path.exists('/content/bundle/processed/glass_merged_seg/manifest.csv'):
    !cp "{DRIVE}/bundle_colab.zip" /content/ && mkdir -p /content/bundle && unzip -q /content/bundle_colab.zip -d /content/bundle
!python -m glass_crack_models.data.relabel_classes
!python -c "from ultralytics.data.utils import check_det_dataset as c; d=c('/content/bundle/processed/glass_merged_linear/data_balanced.yaml'); print(d['names'], d['nc'])"

## Optional checks (first session only)
Unit tests and a 1-epoch smoke run on GPU (results go to `results_smoke.csv`, not the main table).

In [ ]:
!python -m unittest discover -s glass_crack_models/tests -t .
!python -m glass_crack_models.train.train_yolo --config e2_yolo_linear.yaml --smoke --device 0

## Train (one per session)
Each command trains, then evaluates `best` once: threshold frozen on valid (max F1), test, NCUGD, AGDD, failure galleries, and one row appended to `results/results.csv` on Drive. Fill in `--compute-units` from the Colab resources panel after the run, or edit the CSV.

In [ ]:
# E2: YOLO-seg, linear_defect
!python -m glass_crack_models.train.train_yolo --config e2_yolo_linear.yaml --seed 0

In [ ]:
# E1: YOLO-seg, 4 classes (scored as crack OR scratch)
!python -m glass_crack_models.train.train_yolo --config e1_yolo_4class.yaml --seed 0

In [ ]:
# E3: U-Net ResNet34, linear_defect, BCE + Dice (ablations: e3_unet_r34_tversky.yaml, e3_unet_r34_cldice.yaml)
!python -m glass_crack_models.train.train_semseg --config e3_unet_r34.yaml --seed 0

## Re-score, predict, inspect

In [ ]:
# Re-score an existing checkpoint from cached predictions (no GPU needed for scoring)
# !python -m glass_crack_models.eval.run_eval --checkpoint "$OUT_ROOT/runs/E2_s0/weights/best.pt" --exp-id E2 --seed 0 --reuse-cache

# Predict on a folder of photos (threshold from operating_point.json next to the checkpoint)
# !python -m glass_crack_models.infer.predict --checkpoint "$OUT_ROOT/runs/E2_s0/weights/best.pt" --input /content/photos --out "$OUT_ROOT/predictions/E2_s0"

import pandas as pd, os
p = os.path.join(os.environ['OUT_ROOT'], 'results', 'results.csv')
cols = ['exp_id', 'task', 'seed', 'threshold', 'test_recall', 'test_recall_lo', 'test_recall_hi', 'test_precision',
        'test_mask_iou', 'test_fp100', 'ncugd_recall', 'agdd_pair_recall', 'compute_units']
pd.read_csv(p)[cols] if os.path.exists(p) else 'no results yet'